In [ ]:
# --- repo bootstrap (auto-added) ---
# Run paths relative to the repo root and make src/ importable.
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
_SRC = os.path.abspath("src")
if _SRC not in sys.path:
    sys.path.insert(0, _SRC)


In [ ]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

In [ ]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

In [ ]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

In [ ]:
from dotenv import load_dotenv

import os
import openai
from openai import OpenAI
from anthropic import Anthropic
from mistralai import Mistral
import cohere
import google.generativeai as genai
from xai_sdk import Client as XAIClient


load_dotenv()


ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
    "API_KEY_DEEPSEEK": "DeepSeek",
    "API_KEY_GROK": "Grok",
    "API_KEY_ANTHROPIC": "Anthropic",
    "API_KEY_GEMINI": "Gemini",
    "API_KEY_MISTRAL": "Mistral",
    "API_KEY_COHERE": "Cohere",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        print(f"Warning: {name} - API key missing in .env file.")
        continue

API_KEY_OPENAI = os.getenv("API_KEY_OPENAI")
API_KEY_DEEPSEEK = os.getenv("API_KEY_DEEPSEEK")
API_KEY_ANTHROPIC = os.getenv("API_KEY_ANTHROPIC")
API_KEY_GEMINI = os.getenv("API_KEY_GEMINI")
API_KEY_MISTRAL = os.getenv("API_KEY_MISTRAL")
API_KEY_COHERE = os.getenv("API_KEY_COHERE")
API_KEY_GROK = os.getenv("API_KEY_GROK")


backend_to_run = [
    "openai-4.1-mini",
    "openai-4o-mini",
    "mistral-small-2506",
    "mistral-small-2503",
    "anthropic-sonnet",
    "deepseek-v3-chat",
]

backends = {
    "openai-4.1-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4.1-mini-2025-04-14",
        "fname":   "openai_4.1_mini"
    },
    "openai-4o-mini": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_OPENAI),
        "model":   "gpt-4o-mini-2024-07-18",
        "fname":   "openai_4o_mini"
    },

    "deepseek-v3-chat": {
        "provider": "openai",
        "client":  OpenAI(api_key=API_KEY_DEEPSEEK, base_url="https://api.deepseek.com"),
        "model":   "deepseek-chat",
        "fname":   "deepseek_v3"
    },

    "anthropic-sonnet": {
        "provider": "anthropic",
        "client":  Anthropic(api_key=API_KEY_ANTHROPIC),
        "model":   "claude-3-7-sonnet-latest",
        "fname":   "anthropic_3_7_sonnet"
    },

    "gemini-2.5-flash": {
        "provider": "gemini",
        "client":  (genai.configure(api_key=API_KEY_GEMINI) or genai.GenerativeModel("gemini-2.5-flash")),
        "model":   "gemini-2.5-flash",
        "fname":   "google_gemini_2_5_flash"
    },

    "mistral-small-2506": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2506",
        "fname":   "mistral_small_2506"
    },
    "mistral-small-2503": {
        "provider": "mistral",
        "client":  Mistral(api_key=API_KEY_MISTRAL),
        "model":   "mistral-small-2503",
        "fname":   "mistral_small_2503"
    },
}


## Generated Knowledge Prompting (GKP)

In [ ]:
import pandas as pd
import os, json
from tqdm import tqdm

from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from cases.stereotypes_case import stereotypes_case 
from cases.manipulation_case import manipulation_case
from generated_knowledge_prompting import GeneratedKnowledgePrompting
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from data_loader import get_additional_fields

def run_gkp_experiment(case_name: str, backend: str, backends_dict: dict = backends):

    if backend not in backend_to_run:
        print(f"Error: Backend {backend} not in the list.")
        return

    model = backends_dict[backend]["model"]
    client = backends_dict[backend]["client"]
    model_filename = backends_dict[backend]["fname"]


    if case_name == "stereotype":
        case = stereotypes_case
        task_def = stereotype_definition_short_binary
        data = sample_mgsd
        examples_df = sample_examples_mgsd
        label_col = "label"
    elif case_name == "manipulation":
        case = manipulation_case
        task_def = manipulation_definition_short
        data = sample_mentalmanip
        examples_df = sample_examples_mentalmanip
        label_col = "manipulative"
    else:
        raise ValueError(f"Unsupported case: {case_name}")

    gkp = GeneratedKnowledgePrompting(
        case=case,
        client=client,
        model=model,
        task_definition=task_def,
        max_tokens=300,
        knowledge_max_tokens=200,
        examples_df=examples_df,
        knowledge_type="contextual"
    )

    rows = []
    for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"GKP: {case_name}"):
        input_text = row[case.input_col]
        true_label = row[label_col]

        try:
            pred, stats = gkp.classify(input_text)
            pred_mapped = case.label_map.get(pred.strip(), list(case.label_map.values())[-1])
        except Exception as e:
            print(f"Error at {idx}: {e}")
            continue

        additional = get_additional_fields(row, case_name)
        rows.append({
            "sample_id": idx,
            "text": input_text,
            "true_label": true_label,
            "pred_label": pred_mapped,
            "tokens_used": stats["tokens_used"],
            "prompt_tokens": stats["prompt_tokens"],
            "completion_tokens": stats["completion_tokens"],
            "latency": stats["latency"],
            "knowledge": "\n".join(stats.get("generated_knowledge", [])),
            **additional
        })

    df_out = pd.DataFrame(rows)
    output_path = f"results/{model_filename}/gkp/classic/results_{case_name}_gkp.csv"
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    df_out.to_csv(output_path, index=False)
    print(f"✅ Saved {len(df_out)} samples to {output_path}")

    y_true = df_out["true_label"]
    y_pred = df_out["pred_label"]

    if case_name == "manipulation":
        y_true = y_true.astype(int)
        y_pred = y_pred.astype(int)
    else:
        y_true = y_true.astype(str).str.lower().str.strip()
        y_pred = y_pred.astype(str).str.lower().str.strip()

    print("\n=== Classification Report ===")
    print(classification_report(y_true, y_pred))

    print("\n=== Confusion Matrix ===")
    print(pd.DataFrame(confusion_matrix(y_true, y_pred), index=sorted(set(y_true)), columns=sorted(set(y_pred))))

    print(f"\n=== Accuracy: {accuracy_score(y_true, y_pred):.2%}\n")

for backend in backend_to_run:
    run_gkp_experiment("stereotype", backend=backend)
    run_gkp_experiment("manipulation", backend=backend)
